# Install dependencies

In [ ]:
!pip install -q codecarbon torch transformers peft accelerate pillow pypdfium2 opencv-python numpy pandas

# import dependencies & warmup time

In [ ]:
import os
import json
import time
from datetime import datetime
from collections import Counter

import torch
from torch import nn
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model
from PIL import Image
import pypdfium2 as pdfium
import numpy as np
import cv2
import pandas as pd
from codecarbon import EmissionsTracker

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MAIN_OUTPUT_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"
PART1_ROOT = os.path.join(MAIN_OUTPUT_DIR, "phase1_part1_fixed")
os.makedirs(PART1_ROOT, exist_ok=True)

CPU_WARMUP_SECS = 300
GPU_WARMUP_SECS = 300

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str, measure_power_secs: int = 1):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=measure_power_secs,
        log_level="warning",
    )

# Loading the fine-tuned model

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/uml_clip_lora_classifier_N12.pt"

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    task_type="FEATURE_EXTRACTION",
)

class CLIPUMLClassifier(nn.Module):
    def __init__(self, clip_model, num_labels: int):
        super().__init__()
        self.clip = clip_model
        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, pixel_values):
        img_feats = self.clip.get_image_features(pixel_values=pixel_values)
        logits = self.classifier(img_feats.pooler_output if hasattr(img_feats, 'pooler_output') else img_feats)
        return logits

def load_uml_classifier(model_path=MODEL_PATH):
    checkpoint = torch.load(model_path, map_location=device)

    base_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    for p in base_clip.parameters():
        p.requires_grad = False
    base_clip = get_peft_model(base_clip, lora_config)

    num_labels = len(checkpoint["label_to_id"])
    clf = CLIPUMLClassifier(base_clip, num_labels)
    clf.load_state_dict(checkpoint["model_state"])
    clf.to(device)
    clf.eval()

    id_to_label = checkpoint["id_to_label"]
    return clf, id_to_label

# Use clip-vit-base-patch32

In [ ]:
setup_tracking_dir = os.path.join(PART1_ROOT, "setup_tracking")
setup_tracker = make_tracker(
    output_dir=setup_tracking_dir,
    project_name="phase1_part1_setup",
    output_file="phase1_part1_setup_emissions.csv",
    measure_power_secs=1
)

print("PART 1 SETUP TRACKER START")
setup_tracker.start()
setup_t0 = time.time()

uml_classifier, id_to_label_loaded = load_uml_classifier()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("Loaded labels:", id_to_label_loaded)

setup_emissions = setup_tracker.stop()
setup_duration = time.time() - setup_t0
print("PART 1 SETUP TRACKER STOP")

setup_summary = {
    "phase": "phase1_part1",
    "mode": "run_once_only",
    "setup_duration_sec": setup_duration,
    "setup_emissions_kg": setup_emissions,
    "device": device,
    "model_path": MODEL_PATH,
    "clip_model_id": "openai/clip-vit-base-patch32",
    "loaded_labels": id_to_label_loaded
}
save_json(setup_summary, os.path.join(setup_tracking_dir, "phase1_part1_setup_summary.json"))
setup_summary

# Warmup code for CPU and GPU

In [ ]:
def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_part1_clip():
    print("CLIP/GPU warm-up (excluded): 2 tiny inferences...")
    dummy_img = Image.new("RGB", (224, 224), color="white")
    inputs = processor(images=dummy_img, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    _ = uml_classifier(pixel_values)
    _ = uml_classifier(pixel_values)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("CLIP/GPU warm-up finished.\n")

def run_part1_warmup():
    warmup_cpu(CPU_WARMUP_SECS)

    print(f"Part 1 CLIP/GPU warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()
    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_part1_clip()
        time.sleep(3)
    print("Part 1 warm-up completed.\n")

print("\n========== PART 1 WARM-UP START (EXCLUDED) ==========")
run_part1_warmup()
print("========== PART 1 WARM-UP STOP ==========\n")

# UML extraction from design report pdfs

In [ ]:
@torch.no_grad()
def predict_uml_type_pil(pil_img, uml_conf_thresh=0.55):
    inputs = processor(images=pil_img, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    logits = uml_classifier(pixel_values)
    probs = logits.softmax(dim=-1)[0]
    pred_id = int(probs.argmax().item())
    label = id_to_label_loaded[pred_id]
    conf = float(probs[pred_id].item())

    if label != "non_uml" and conf < uml_conf_thresh:
        label = "non_uml"

    return label, conf

def extract_page_images(pdf_path, out_dir, scale=2.0):
    os.makedirs(out_dir, exist_ok=True)
    pdf = pdfium.PdfDocument(pdf_path)
    paths = []
    for i in range(len(pdf)):
        page = pdf[i]
        bitmap = page.render(scale=scale)
        img = bitmap.to_pil()
        out_path = os.path.join(out_dir, f"page_{i+1:03d}.png")
        img.save(out_path)
        paths.append(out_path)
    print(f"✅ Extracted {len(paths)} page images from {os.path.basename(pdf_path)}")
    return paths

UML_MIN_CONF = 0.60

def classify_pdf_pages_and_save_uml(pdf_page_paths, uml_only_pdf_path):
    per_page = []
    uml_subtype_counts = Counter()
    non_uml_count = 0

    print("===== PER-PAGE CLASSIFICATION =====")
    for idx, path in enumerate(pdf_page_paths):
        pil_img = Image.open(path)
        label, conf = predict_uml_type_pil(pil_img)
        is_uml = (label != "non_uml") and (conf >= UML_MIN_CONF)

        print_label = label
        if not is_uml and label != "non_uml":
            print_label = f"{label} (low_conf)"

        print(f"{os.path.basename(path)}  ->  {print_label:18s}  (conf = {conf:.3f})")

        if is_uml:
            uml_subtype_counts[label] += 1
        else:
            non_uml_count += 1

        per_page.append(
            {
                "page_idx": idx,
                "path": path,
                "raw_label": label,
                "conf": conf,
                "is_uml": is_uml,
            }
        )

    total_uml = sum(uml_subtype_counts.values())
    total_pages = len(pdf_page_paths)

    print("\n===== LEVEL-1 SUMMARY (UML vs non-UML) =====")
    print(f"UML diagrams : {total_uml}")
    print(f"Non-UML      : {non_uml_count}")
    print(f"Total pages  : {total_pages}")

    print("\n===== LEVEL-2 SUMMARY (UML subtypes) =====")
    for uml_type, cnt in sorted(uml_subtype_counts.items()):
        print(f"{uml_type:15s} : {cnt}")

    uml_paths = [p["path"] for p in per_page if p["is_uml"]]
    if not uml_paths:
        print("\n⚠️ No confident UML pages; UML-only PDF not created.")
        return per_page, uml_subtype_counts, non_uml_count, None

    uml_images = [Image.open(p).convert("RGB") for p in uml_paths]
    first = uml_images[0]
    rest = uml_images[1:]
    first.save(uml_only_pdf_path, save_all=True, append_images=rest)
    print(f"\n✅ Saved UML-only pages PDF: {uml_only_pdf_path}")

    return per_page, uml_subtype_counts, non_uml_count, uml_only_pdf_path

def extract_diagram_crops_from_pages(
    pdf_path: str,
    out_dir: str,
    scale: float = 3.0,
    top_cut_ratio: float = 0.20,
    bottom_cut_ratio: float = 0.0,
    min_area_ratio: float = 0.003,
    max_area_ratio: float = 0.70,
    margin: int = 20,
    extra_top_ratio: float = 0.15,
    extra_bottom_ratio: float = 0.25,
):
    os.makedirs(out_dir, exist_ok=True)
    pdf = pdfium.PdfDocument(pdf_path)
    crop_paths = []

    for page_idx in range(len(pdf)):
        page = pdf[page_idx]
        bitmap = page.render(scale=scale)
        pil_img = bitmap.to_pil().convert("RGB")
        page_w, page_h = pil_img.size

        y_top = int(page_h * top_cut_ratio)
        y_bottom = int(page_h * (1.0 - bottom_cut_ratio))
        roi = pil_img.crop((0, y_top, page_w, y_bottom))

        gray = np.array(roi.convert("L"))
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blurred, 50, 150)

        kernel = np.ones((9, 9), np.uint8)
        dilated = cv2.dilate(edges, kernel, iterations=2)

        contours, _ = cv2.findContours(
            dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        roi_h, roi_w = gray.shape
        roi_area = float(roi_h * roi_w)

        print(f"Page {page_idx+1:03d}: {len(contours)} raw contours in ROI")

        page_crops = 0
        contours = sorted(contours, key=cv2.contourArea, reverse=True)

        for ci, cnt in enumerate(contours):
            area = cv2.contourArea(cnt)
            if area < min_area_ratio * roi_area:
                continue
            if area > max_area_ratio * roi_area:
                continue

            x, y, cw, ch = cv2.boundingRect(cnt)

            if cw < 80 or ch < 80:
                continue

            extra_top    = int(extra_top_ratio * ch)
            extra_bottom = int(extra_bottom_ratio * ch)

            x0 = max(x - margin, 0)
            y0 = max(y + y_top - margin - extra_top, 0)
            x1 = min(x + cw + margin, page_w)
            y1 = min(y + y_top + ch + margin + extra_bottom, page_h)

            crop = pil_img.crop((x0, y0, x1, y1))

            out_path = os.path.join(
                out_dir, f"page_{page_idx+1:03d}_crop_{page_crops}.png"
            )
            crop.save(out_path)
            crop_paths.append(out_path)
            page_crops += 1

        print(f"↳ Page {page_idx+1:03d}: saved {page_crops} crop(s).")

    print(f"\nTotal crops saved from {os.path.basename(pdf_path)}: {len(crop_paths)}")
    return crop_paths

def crops_to_pdf(crop_paths, out_pdf_path):
    if not crop_paths:
        print("⚠️ No crops found, not creating PDF.")
        return

    crop_paths = sorted(crop_paths)
    images = [Image.open(p).convert("RGB") for p in crop_paths]
    first, rest = images[0], images[1:]
    first.save(out_pdf_path, save_all=True, append_images=rest)
    print(f"✅ Saved UML-only cropped diagrams PDF: {out_pdf_path}")

# input path

In [ ]:
REPORTS_DIR = "/content/drive/MyDrive/All_Projects_Reports"

PAGE_IMAGES_ROOT   = os.path.join(PART1_ROOT, "report_pages_all")
UML_PAGES_PDF_DIR  = os.path.join(PART1_ROOT, "uml_pages_pdfs")
UML_CROPS_PNG_ROOT = os.path.join(PART1_ROOT, "uml_crops_png")
UML_CROPS_PDF_DIR  = os.path.join(PART1_ROOT, "uml_crops_pdfs")
LOGS_DIR           = os.path.join(PART1_ROOT, "logs")
TRACKING_DIR       = os.path.join(PART1_ROOT, "inference_tracking")

os.makedirs(PAGE_IMAGES_ROOT, exist_ok=True)
os.makedirs(UML_PAGES_PDF_DIR, exist_ok=True)
os.makedirs(UML_CROPS_PNG_ROOT, exist_ok=True)
os.makedirs(UML_CROPS_PDF_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(TRACKING_DIR, exist_ok=True)

print("REPORTS_DIR:", REPORTS_DIR)
print("PART1_ROOT:", PART1_ROOT)
print("UML_PAGES_PDF_DIR:", UML_PAGES_PDF_DIR)

# inference tracking

In [ ]:
inference_tracker = make_tracker(
    output_dir=TRACKING_DIR,
    project_name="phase1_part1_inference",
    output_file="phase1_part1_inference_emissions.csv",
    measure_power_secs=1
)

print("PART 1 INFERENCE TRACKER START")
inference_tracker.start()
infer_t0 = time.time()

pdf_files = [
    f for f in os.listdir(REPORTS_DIR)
    if f.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s) in {REPORTS_DIR}:", pdf_files)

overall_summary_rows = []
token_rows = []

for pdf_name in pdf_files:
    print("\n" + "="*80)
    print(f"Processing PDF: {pdf_name}")
    print("="*80)

    pdf_path = os.path.join(REPORTS_DIR, pdf_name)
    base = os.path.splitext(pdf_name)[0]

    page_img_dir = os.path.join(PAGE_IMAGES_ROOT, base)
    page_images = extract_page_images(pdf_path, out_dir=page_img_dir, scale=2.0)

    uml_pages_pdf_path = os.path.join(UML_PAGES_PDF_DIR, f"{base}_uml_pages.pdf")
    per_page_results, uml_counts, non_uml_count, uml_pages_pdf_path_ret = \
        classify_pdf_pages_and_save_uml(page_images, uml_pages_pdf_path)

    crop_count = 0
    crops_pdf_path = None

    if uml_pages_pdf_path_ret is None or not os.path.exists(uml_pages_pdf_path_ret):
        print(f"⚠️ Skipping cropping for {pdf_name} (no UML pages detected).")
    else:
        crop_dir = os.path.join(UML_CROPS_PNG_ROOT, base)
        crops_pdf_path = os.path.join(UML_CROPS_PDF_DIR, f"{base}_uml_diagrams_cropped.pdf")

        crop_paths = extract_diagram_crops_from_pages(
            uml_pages_pdf_path_ret,
            crop_dir,
            scale=3.0,
            top_cut_ratio=0.20,
            bottom_cut_ratio=0.0,
            min_area_ratio=0.003,
            max_area_ratio=0.70,
            margin=20,
            extra_top_ratio=0.15,
            extra_bottom_ratio=0.60,
        )

        crop_count = len(crop_paths)
        crops_to_pdf(crop_paths, crops_pdf_path)

    summary_row = {
        "report_pdf": pdf_name,
        "total_pages": len(page_images),
        "uml_pages_detected": int(sum(1 for x in per_page_results if x["is_uml"])),
        "non_uml_pages": int(non_uml_count),
        "uml_subtype_counts": dict(uml_counts),
        "uml_pages_pdf_path": uml_pages_pdf_path_ret,
        "crops_pdf_path": crops_pdf_path,
        "crop_count": crop_count,
    }
    overall_summary_rows.append(summary_row)
    save_json(summary_row, os.path.join(LOGS_DIR, f"{base}_summary.json"))

    token_rows.append({
        "phase": "phase1_part1",
        "report": pdf_name,
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "note": "No text generation in Part 1"
    })

inference_emissions = inference_tracker.stop()
inference_duration = time.time() - infer_t0
print("PART 1 INFERENCE TRACKER STOP")

print("\n✅ Finished processing all PDFs.")

# Save all the outputs

In [ ]:
summary_csv_path = os.path.join(PART1_ROOT, "phase1_part1_summary.csv")
pd.DataFrame(overall_summary_rows).to_csv(summary_csv_path, index=False)

token_csv_path = os.path.join(PART1_ROOT, "phase1_part1_token_counts.csv")
pd.DataFrame(token_rows).to_csv(token_csv_path, index=False)

inference_summary = {
    "phase": "phase1_part1",
    "mode": "run_once_only",
    "inference_duration_sec": inference_duration,
    "inference_emissions_kg": inference_emissions,
    "reports_dir": REPORTS_DIR,
    "uml_pages_pdf_dir": UML_PAGES_PDF_DIR,
    "uml_crops_pdf_dir": UML_CROPS_PDF_DIR,
    "summary_csv": summary_csv_path,
    "token_csv": token_csv_path
}
save_json(inference_summary, os.path.join(TRACKING_DIR, "phase1_part1_inference_summary.json"))

final_summary = {
    "phase": "phase1_part1",
    "mode": "fixed_once",
    "setup_summary": setup_summary,
    "inference_summary": inference_summary,
    "downstream_input_for_part2": UML_PAGES_PDF_DIR
}
save_json(final_summary, os.path.join(PART1_ROOT, "phase1_part1_final_summary.json"))

print("\nSaved Part 1 root:", PART1_ROOT)
print("UML PDF dir for Part 2:", UML_PAGES_PDF_DIR)
print("Summary CSV:", summary_csv_path)
print("Token CSV:", token_csv_path)